# Task 2 — Unsupervised Domain Adaptation (PACS)
Restructured for idempotent re-runs: every section is self-contained top to bottom.
**Run order matters** — each section below depends only on sections above it, never on
anything from a previous Kaggle session's memory. If your session restarts, just
Run All from the top; nothing here assumes leftover state.


## 0. Setup — imports, seed, BatchNorm-freeze utility
Run this first, always. `set_seed` is called once here; every training cell below
re-calls it before building its model so results stay reproducible under seed 6304
regardless of what ran earlier in the session.


In [1]:
import os
import copy
import random
import numpy as np

import torch
import torch.nn as nn
from torch.autograd import Function
from torch.optim import AdamW
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

def set_seed(seed=6304):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(6304)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def freeze_bn_stats(m):
    """Freezes BatchNorm running statistics at their pretrained values but
    keeps scale (gamma) and bias (beta) trainable. Call model.apply(freeze_bn_stats)
    AFTER model.train() every epoch, for every method in Tasks 2 and 3."""
    if m.__class__.__name__.find('BatchNorm') != -1:
        m.eval()

print("Setup complete.")


Using device: cuda
Setup complete.


## 1. Data — PACS loaders (Photo / Art Painting / Cartoon source, Sketch target)
Stratified 80/20 source splits with seed 6304, per spec. Source train batches give
8 examples/domain (24 total); target batches give 24 examples, matching total source
and target batch sizes as required.


In [2]:
# Auto-locate the dataset root in Kaggle's mounted input folder
base_dir = ""
for root, dirs, files in os.walk('/kaggle/input/'):
    if {'photo', 'art_painting', 'cartoon', 'sketch'}.issubset({d.lower() for d in dirs}):
        base_dir = root
        break

print(f"Dataset located at: {base_dir}")

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_stratified_split(dataset, seed=6304, test_size=0.2):
    labels = dataset.targets
    train_idx, val_idx = train_test_split(
        range(len(labels)), test_size=test_size, stratify=labels, random_state=seed
    )
    return train_idx, val_idx

# Source loaders: Photo, Art Painting, Cartoon
sources = ['photo', 'art_painting', 'cartoon']
source_loaders = {}

for domain in sources:
    domain_path = os.path.join(base_dir, domain)
    full_train_ds = ImageFolder(root=domain_path, transform=train_transform)
    full_val_ds = ImageFolder(root=domain_path, transform=eval_transform)

    train_idx, val_idx = get_stratified_split(full_train_ds, seed=6304)

    train_subset = Subset(full_train_ds, train_idx)
    val_subset = Subset(full_val_ds, val_idx)

    source_loaders[domain] = {
        'train': DataLoader(train_subset, batch_size=8, shuffle=True, drop_last=True),
        'val': DataLoader(val_subset, batch_size=32, shuffle=False)
    }

# Target loader: Sketch
target_path = os.path.join(base_dir, 'sketch')
target_ds_train = ImageFolder(root=target_path, transform=train_transform)
target_ds_eval = ImageFolder(root=target_path, transform=eval_transform)

target_loader_train = DataLoader(target_ds_train, batch_size=24, shuffle=True, drop_last=True)
target_loader_eval = DataLoader(target_ds_eval, batch_size=32, shuffle=False)

print("Data loaders ready:")
for d in sources:
    print(f"  {d}: {len(source_loaders[d]['train'].dataset)} train / {len(source_loaders[d]['val'].dataset)} val")
print(f"  sketch (target): {len(target_ds_train)} images")


Dataset located at: /kaggle/input/datasets/nickfratto/pacs-dataset/pacs_data/pacs_data
Data loaders ready:
  photo: 1336 train / 334 val
  art_painting: 1638 train / 410 val
  cartoon: 1875 train / 469 val
  sketch (target): 3929 images


## 2. Shared evaluation utility
Works for ERM, DANN, and CDAN alike — every model here returns plain class logits
from `forward(x)` when called without `alpha`.


In [3]:
def evaluate_loader(model, loader):
    """Computes Top-1 Accuracy and Macro-F1 for a given dataloader."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return acc * 100, macro_f1 * 100

def report_domains(model, tag):
    print(f"--- {tag} Evaluation ---")
    for domain in ['photo', 'art_painting', 'cartoon']:
        acc, f1 = evaluate_loader(model, source_loaders[domain]['val'])
        print(f"Source Val [{domain.capitalize()}]: Accuracy = {acc:.2f}% | Macro-F1 = {f1:.2f}%")
    target_acc, target_f1 = evaluate_loader(model, target_loader_eval)
    print(f"Target [Sketch]: Accuracy = {target_acc:.2f}% | Macro-F1 = {target_f1:.2f}%")
    return target_acc, target_f1


## 3. Source-Only ERM baseline
This checkpoint is also reused unchanged as the Task 3 ERM baseline — don't retrain
it differently there.


In [4]:
set_seed(6304)

erm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
erm_model.fc = nn.Linear(erm_model.fc.in_features, 7)
erm_model.to(device)

optimizer = AdamW(erm_model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

max_epochs = 30
patience = 5
best_mean_f1 = -1
epochs_no_improve = 0
best_state = None

photo_loader = source_loaders['photo']['train']
art_loader = source_loaders['art_painting']['train']
cartoon_loader = source_loaders['cartoon']['train']

print("Starting Source-Only ERM Training...")

for epoch in range(max_epochs):
    erm_model.train()
    erm_model.apply(freeze_bn_stats)

    total_loss = 0
    for (img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c) in zip(photo_loader, art_loader, cartoon_loader):
        images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
        labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)

        optimizer.zero_grad()
        outputs = erm_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    erm_model.eval()
    f1s = [evaluate_loader(erm_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
    mean_f1 = np.mean(f1s)
    print(f"Epoch {epoch+1}/{max_epochs} | Loss: {total_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        best_state = copy.deepcopy(erm_model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
            break

torch.save(best_state, '/kaggle/working/source_only_erm.pth')
print(f"ERM training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved.")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s] 


Starting Source-Only ERM Training...
Epoch 1/30 | Loss: 91.2043 | Mean Source Val F1: 91.02%
Epoch 2/30 | Loss: 41.0888 | Mean Source Val F1: 90.28%
Epoch 3/30 | Loss: 30.3199 | Mean Source Val F1: 91.93%
Epoch 4/30 | Loss: 27.5253 | Mean Source Val F1: 90.89%
Epoch 5/30 | Loss: 19.0759 | Mean Source Val F1: 91.99%
Epoch 6/30 | Loss: 17.2236 | Mean Source Val F1: 89.06%
Epoch 7/30 | Loss: 13.5507 | Mean Source Val F1: 93.95%
Epoch 8/30 | Loss: 14.8458 | Mean Source Val F1: 91.46%
Epoch 9/30 | Loss: 14.1339 | Mean Source Val F1: 93.99%
Epoch 10/30 | Loss: 12.1688 | Mean Source Val F1: 91.41%
Epoch 11/30 | Loss: 9.9160 | Mean Source Val F1: 93.92%
Epoch 12/30 | Loss: 10.8674 | Mean Source Val F1: 92.15%
Epoch 13/30 | Loss: 11.6728 | Mean Source Val F1: 92.36%
Epoch 14/30 | Loss: 6.2008 | Mean Source Val F1: 92.78%
Early stopping at epoch 14 (best mean F1: 93.99%)
ERM training complete. Best checkpoint (mean source F1=93.99%) saved.


In [5]:
# --- ERM evaluation ---
erm_model.load_state_dict(torch.load('/kaggle/working/source_only_erm.pth'))
erm_model.to(device)
erm_target_acc, erm_target_f1 = report_domains(erm_model, "Source-Only ERM")


--- Source-Only ERM Evaluation ---
Source Val [Photo]: Accuracy = 96.41% | Macro-F1 = 95.82%
Source Val [Art_painting]: Accuracy = 92.20% | Macro-F1 = 91.72%
Source Val [Cartoon]: Accuracy = 94.46% | Macro-F1 = 94.43%
Target [Sketch]: Accuracy = 72.46% | Macro-F1 = 74.41%


## 4. DANN — Adversarial Alignment
### 4a. Architecture (Gradient Reversal Layer + model)


In [6]:
class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None


class DANN_ResNet(nn.Module):
    def __init__(self, num_classes=7):
        super(DANN_ResNet, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        in_features = resnet.fc.in_features

        self.class_classifier = nn.Linear(in_features, num_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )

    def forward(self, x, alpha=None):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        class_preds = self.class_classifier(features)

        if alpha is not None:
            reversed_features = GradientReversalFn.apply(features, alpha)
            domain_preds = self.domain_classifier(reversed_features)
            return class_preds, domain_preds

        return class_preds

print("DANN architecture defined.")


DANN architecture defined.


### 4b. Training
AdamW with `eps=1e-4` (loosened from default `1e-8` for GRL stability), grad-norm
clipping, seed reset, per-batch diagnostic logging, and checkpoint selection with
early stopping on mean source-validation macro-F1.


In [7]:
set_seed(6304)

dann_model = DANN_ResNet()
dann_model.to(device)

optimizer = AdamW(dann_model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
class_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.CrossEntropyLoss()

max_epochs = 30
patience = 5
best_mean_f1 = -1
epochs_no_improve = 0
best_state = None

photo_loader = source_loaders['photo']['train']
art_loader = source_loaders['art_painting']['train']
cartoon_loader = source_loaders['cartoon']['train']
num_batches = min(len(photo_loader), len(art_loader), len(cartoon_loader))

print("Starting DANN Adversarial Training...")

for epoch in range(max_epochs):
    dann_model.train()
    dann_model.apply(freeze_bn_stats)

    target_iter = iter(target_loader_train)
    total_class_loss, total_domain_loss = 0, 0

    for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
        zip(photo_loader, art_loader, cartoon_loader)):

        try:
            img_t, _ = next(target_iter)
        except StopIteration:
            target_iter = iter(target_loader_train)
            img_t, _ = next(target_iter)

        p = float(batch_idx + epoch * num_batches) / (max_epochs * num_batches)
        alpha = 2. / (1. + np.exp(-10 * p)) - 1

        src_images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
        src_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)
        src_domain_labels = torch.zeros(src_images.size(0), dtype=torch.long).to(device)

        src_class_preds, src_domain_preds = dann_model(src_images, alpha=alpha)
        loss_class = class_criterion(src_class_preds, src_labels)
        loss_domain_src = domain_criterion(src_domain_preds, src_domain_labels)

        tgt_images = img_t.to(device)
        tgt_domain_labels = torch.ones(tgt_images.size(0), dtype=torch.long).to(device)
        _, tgt_domain_preds = dann_model(tgt_images, alpha=alpha)
        loss_domain_tgt = domain_criterion(tgt_domain_preds, tgt_domain_labels)

        loss_domain = loss_domain_src + loss_domain_tgt
        total_loss = loss_class + loss_domain

        optimizer.zero_grad()
        total_loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(dann_model.parameters(), max_norm=1.0)

        if not torch.isfinite(grad_norm):
            optimizer.zero_grad()
            if batch_idx % 20 == 0:
                print(f"  batch {batch_idx}: SKIPPED (non-finite grad)")
            continue

        optimizer.step()

        total_class_loss += loss_class.item()
        total_domain_loss += loss_domain.item()

        if batch_idx % 20 == 0:
            print(f"  batch {batch_idx}: class={loss_class.item():.3f}, "
                  f"domain={loss_domain.item():.3f}, grad_norm={grad_norm.item():.3f}, alpha={alpha:.3f}")

    dann_model.eval()
    f1s = [evaluate_loader(dann_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
    mean_f1 = np.mean(f1s)

    print(f"Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
          f"Domain Loss: {total_domain_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        best_state = copy.deepcopy(dann_model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
            break

torch.save(best_state, '/kaggle/working/dann_model.pth')
print(f"DANN training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved.")


Starting DANN Adversarial Training...
  batch 0: class=2.288, domain=1.444, grad_norm=15.458, alpha=0.000
  batch 20: class=1.029, domain=1.025, grad_norm=16.156, alpha=0.020
  batch 40: class=0.365, domain=0.977, grad_norm=12.226, alpha=0.040
  batch 60: class=0.534, domain=1.280, grad_norm=17.751, alpha=0.060
  batch 80: class=0.248, domain=4.711, grad_norm=64.724, alpha=0.080
  batch 100: class=0.259, domain=8.696, grad_norm=105.746, alpha=0.099
  batch 120: class=0.609, domain=22.062, grad_norm=234.658, alpha=0.119
  batch 140: class=3.066, domain=63.521, grad_norm=1312.161, alpha=0.139
  batch 160: class=3.187, domain=59.113, grad_norm=1247.185, alpha=0.158
Epoch 1/30 | Class Loss: 279.6902 | Domain Loss: 2964.3142 | Mean Source Val F1: 63.44%
  batch 0: class=3.838, domain=32.445, grad_norm=1389.725, alpha=0.165
  batch 20: class=11.527, domain=203.461, grad_norm=5219.084, alpha=0.184
  batch 40: class=14.893, domain=206.398, grad_norm=4246.522, alpha=0.204
  batch 60: class=29.8

In [8]:
# --- DANN evaluation ---
dann_model.load_state_dict(torch.load('/kaggle/working/dann_model.pth'))
dann_model.to(device)
dann_target_acc, dann_target_f1 = report_domains(dann_model, "DANN")


--- DANN Evaluation ---
Source Val [Photo]: Accuracy = 79.64% | Macro-F1 = 73.59%
Source Val [Art_painting]: Accuracy = 66.10% | Macro-F1 = 61.09%
Source Val [Cartoon]: Accuracy = 56.93% | Macro-F1 = 55.65%
Target [Sketch]: Accuracy = 30.54% | Macro-F1 = 14.34%


## 5. CDAN — Class-Conditional Adversarial Alignment
### 5a. Architecture
Same GRL, same discriminator hidden width/dropout/schedule as DANN. The only
structural difference: the discriminator sees `g(x) = vec(f ⊗ p)` instead of
raw features `f`. No detaching `f` or `p`, per spec.


In [9]:
class CDAN_ResNet(nn.Module):
    def __init__(self, num_classes=7):
        super(CDAN_ResNet, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        in_features = resnet.fc.in_features
        self.num_classes = num_classes

        self.class_classifier = nn.Linear(in_features, num_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(in_features * num_classes, 256),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )

    def forward(self, x, alpha=None):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        class_preds = self.class_classifier(features)

        if alpha is not None:
            probs = torch.softmax(class_preds, dim=1)
            g = torch.bmm(probs.unsqueeze(2), features.unsqueeze(1))  # (batch, num_classes, in_features)
            g = g.view(g.size(0), -1)
            reversed_g = GradientReversalFn.apply(g, alpha)
            domain_preds = self.domain_classifier(reversed_g)
            return class_preds, domain_preds

        return class_preds

print("CDAN architecture defined.")


CDAN architecture defined.


### 5b. Training


In [ ]:
set_seed(6304)

cdan_model = CDAN_ResNet()
cdan_model.to(device)

optimizer = AdamW(cdan_model.parameters(), lr=1e-4, weight_decay=1e-4, eps=1e-4)
class_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.CrossEntropyLoss()

max_epochs = 30
patience = 5
best_mean_f1 = -1
epochs_no_improve = 0
best_state = None

photo_loader = source_loaders['photo']['train']
art_loader = source_loaders['art_painting']['train']
cartoon_loader = source_loaders['cartoon']['train']
num_batches = min(len(photo_loader), len(art_loader), len(cartoon_loader))

print("Starting CDAN Adversarial Training...")

for epoch in range(max_epochs):
    cdan_model.train()
    cdan_model.apply(freeze_bn_stats)

    target_iter = iter(target_loader_train)
    total_class_loss, total_domain_loss = 0, 0

    for batch_idx, ((img_p, lbl_p), (img_a, lbl_a), (img_c, lbl_c)) in enumerate(
        zip(photo_loader, art_loader, cartoon_loader)):

        try:
            img_t, _ = next(target_iter)
        except StopIteration:
            target_iter = iter(target_loader_train)
            img_t, _ = next(target_iter)

        p = float(batch_idx + epoch * num_batches) / (max_epochs * num_batches)
        alpha = 2. / (1. + np.exp(-10 * p)) - 1

        src_images = torch.cat([img_p, img_a, img_c], dim=0).to(device)
        src_labels = torch.cat([lbl_p, lbl_a, lbl_c], dim=0).to(device)
        src_domain_labels = torch.zeros(src_images.size(0), dtype=torch.long).to(device)

        src_class_preds, src_domain_preds = cdan_model(src_images, alpha=alpha)
        loss_class = class_criterion(src_class_preds, src_labels)
        loss_domain_src = domain_criterion(src_domain_preds, src_domain_labels)

        tgt_images = img_t.to(device)
        tgt_domain_labels = torch.ones(tgt_images.size(0), dtype=torch.long).to(device)
        _, tgt_domain_preds = cdan_model(tgt_images, alpha=alpha)
        loss_domain_tgt = domain_criterion(tgt_domain_preds, tgt_domain_labels)

        loss_domain = loss_domain_src + loss_domain_tgt
        total_loss = loss_class + loss_domain

        optimizer.zero_grad()
        total_loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(cdan_model.parameters(), max_norm=1.0)

        if not torch.isfinite(grad_norm):
            optimizer.zero_grad()
            if batch_idx % 20 == 0:
                print(f"  batch {batch_idx}: SKIPPED (non-finite grad)")
            continue

        optimizer.step()

        total_class_loss += loss_class.item()
        total_domain_loss += loss_domain.item()

        if batch_idx % 20 == 0:
            print(f"  batch {batch_idx}: class={loss_class.item():.3f}, "
                  f"domain={loss_domain.item():.3f}, grad_norm={grad_norm.item():.3f}, alpha={alpha:.3f}")

    cdan_model.eval()
    f1s = [evaluate_loader(cdan_model, source_loaders[d]['val'])[1] for d in ['photo', 'art_painting', 'cartoon']]
    mean_f1 = np.mean(f1s)

    print(f"Epoch {epoch+1}/{max_epochs} | Class Loss: {total_class_loss:.4f} | "
          f"Domain Loss: {total_domain_loss:.4f} | Mean Source Val F1: {mean_f1:.2f}%")

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        best_state = copy.deepcopy(cdan_model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (best mean F1: {best_mean_f1:.2f}%)")
            break

torch.save(best_state, '/kaggle/working/cdan_model.pth')
print(f"CDAN training complete. Best checkpoint (mean source F1={best_mean_f1:.2f}%) saved.")


Starting CDAN Adversarial Training...
  batch 0: class=2.321, domain=1.413, grad_norm=13.803, alpha=0.000
  batch 20: class=0.707, domain=0.932, grad_norm=18.571, alpha=0.020
  batch 40: class=0.447, domain=0.663, grad_norm=12.640, alpha=0.040
  batch 60: class=0.501, domain=0.580, grad_norm=13.227, alpha=0.060
  batch 80: class=0.527, domain=0.880, grad_norm=13.057, alpha=0.080
  batch 100: class=0.824, domain=1.024, grad_norm=28.328, alpha=0.099
  batch 120: class=0.144, domain=1.079, grad_norm=15.692, alpha=0.119
  batch 140: class=1.120, domain=1.654, grad_norm=37.626, alpha=0.139
  batch 160: class=0.586, domain=2.764, grad_norm=45.835, alpha=0.158
Epoch 1/30 | Class Loss: 104.9231 | Domain Loss: 250.0542 | Mean Source Val F1: 83.56%
  batch 0: class=0.238, domain=2.351, grad_norm=29.834, alpha=0.165
  batch 20: class=0.454, domain=1.675, grad_norm=30.051, alpha=0.184
  batch 40: class=0.354, domain=2.035, grad_norm=33.021, alpha=0.204
  batch 60: class=0.429, domain=1.219, grad_n

In [ ]:
# --- CDAN evaluation ---
cdan_model.load_state_dict(torch.load('/kaggle/working/cdan_model.pth'))
cdan_model.to(device)
cdan_target_acc, cdan_target_f1 = report_domains(cdan_model, "CDAN")


## 6. Summary table
Quick sanity check across methods before moving on to the domain-separability
diagnostic and per-class analysis.


In [ ]:
print(f"{'Method':<12}{'Sketch Acc':>12}{'Sketch F1':>12}")
print(f"{'ERM':<12}{erm_target_acc:>11.2f}%{erm_target_f1:>11.2f}%")
print(f"{'DANN':<12}{dann_target_acc:>11.2f}%{dann_target_f1:>11.2f}%")
print(f"{'CDAN':<12}{cdan_target_acc:>11.2f}%{cdan_target_f1:>11.2f}%")
